# 第8章 可视化与科研图表

这个 notebook 用行星轨道示例数据说明科学图表的最小规范：读数据、检查量级、明确坐标轴和单位、比较线性视角与对数视角，并在安装了 `matplotlib` 时导出图像。重点不是“画出一张图”，而是让图真正可解释。

## 学习目标

- 理解科学图表的最小要素：标题、坐标轴、单位、图例
- 学会从小型教学数据生成一张可解释的图
- 理解为什么对数坐标有时比线性坐标更有信息
- 建立“图表是论证的一部分”而不是装饰的意识

In [ ]:
from __future__ import annotations

import csv
import math
import platform
from pathlib import Path

DATA_PATH = Path('../../data/small/planetary_orbits_demo.csv')
with DATA_PATH.open(newline='', encoding='utf-8') as handle:
    rows = list(csv.DictReader(handle))

planet_names = [row['planet'] for row in rows]
semi_major_axes = [float(row['semi_major_axis_au']) for row in rows]
orbital_periods = [float(row['orbital_period_years']) for row in rows]

print('数据点数量:', len(semi_major_axes))
print('前两个样本:', list(zip(planet_names, semi_major_axes, orbital_periods))[:2])
print('Python version:', platform.python_version())


## 先做数据检查，而不是马上画图

科研图表的第一步往往不是调颜色，而是确认量级、范围和变量关系是否合理。

In [ ]:
print('semi-major axis range [au]:', (min(semi_major_axes), max(semi_major_axes)))
print('orbital period range [yr]:', (min(orbital_periods), max(orbital_periods)))
kepler_ratios = [round((period ** 2) / (axis ** 3), 3) for axis, period in zip(semi_major_axes, orbital_periods)]
print('P^2 / a^3 ratios:', kepler_ratios)
print('mean P^2 / a^3 =', round(sum((period ** 2) / (axis ** 3) for axis, period in zip(semi_major_axes, orbital_periods)) / len(rows), 3))


## 先选图表类型

图表类型应当由问题决定：比较两个连续变量时用散点图，沿时间或波长变化时用折线图，检查样本分布时用直方图，检查模型遗漏结构时用残差图。本章的行星例子强调两个连续变量的幂律关系，所以散点图和 log-log 图最合适。

## 线性视角与对数视角

同一组数据在不同坐标尺度下会强调不同结构。下面先在数值层面比较线性和对数坐标。

In [ ]:
log_axes = [round(math.log10(value), 3) for value in semi_major_axes]
log_periods = [round(math.log10(value), 3) for value in orbital_periods]
print('log10(a) first four =', log_axes[:4])
print('log10(P) first four =', log_periods[:4])
print('On log scales, Kepler-like power-law structure tends to become more linear.')


## 残差也应该被画出来

如果我们把开普勒第三定律的教学近似写成 $P \approx a^{3/2}$，就可以计算每个行星相对于这个模型的残差。残差不是附属品，它常常比主图更容易暴露模型、单位或数据问题。

In [ ]:
model_periods = [axis ** 1.5 for axis in semi_major_axes]
period_residuals = [observed - model for observed, model in zip(orbital_periods, model_periods)]

for name, observed, model, residual in zip(planet_names, orbital_periods, model_periods, period_residuals):
    print(f'{name:7s} observed={observed:8.3f} model={model:8.3f} residual={residual:8.3f}')

max_abs_residual = max(abs(value) for value in period_residuals)
print('max absolute residual [yr] =', round(max_abs_residual, 3))


## 如果安装了 matplotlib，就导出图表

这里同时导出一张线性坐标图和一张对数坐标图。若环境里没有 `matplotlib`，下面也只会给出清楚提示。

In [ ]:
figure_dir = Path('../../figures/generated')
linear_path = figure_dir / 'ch08_orbital_period_vs_axis_linear.png'
linear_pdf_path = figure_dir / 'ch08_orbital_period_vs_axis_linear.pdf'
log_path = figure_dir / 'ch08_orbital_period_vs_axis_loglog.png'
residual_path = figure_dir / 'ch08_orbital_period_residuals.png'

try:
    import matplotlib.pyplot as plt
except ModuleNotFoundError:
    print('matplotlib 未安装；已跳过绘图。')
else:
    figure_dir.mkdir(parents=True, exist_ok=True)

    fig, ax = plt.subplots(figsize=(6, 4))
    ax.scatter(semi_major_axes, orbital_periods, color='tab:blue')
    for name, x_value, y_value in zip(planet_names, semi_major_axes, orbital_periods):
        ax.annotate(name, (x_value, y_value), fontsize=8, xytext=(4, 4), textcoords='offset points')
    ax.set_title('Orbital Period vs. Semi-major Axis')
    ax.set_xlabel('Semi-major axis [au]')
    ax.set_ylabel('Orbital period [yr]')
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(linear_path, dpi=150)
    fig.savefig(linear_pdf_path)
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(6, 4))
    ax.scatter(semi_major_axes, orbital_periods, color='tab:orange')
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_title('Orbital Period vs. Semi-major Axis (log-log)')
    ax.set_xlabel('Semi-major axis [au]')
    ax.set_ylabel('Orbital period [yr]')
    ax.grid(alpha=0.3, which='both')
    fig.tight_layout()
    fig.savefig(log_path, dpi=150)
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(6, 3))
    ax.axhline(0.0, color='black', linewidth=1, alpha=0.6)
    ax.scatter(semi_major_axes, period_residuals, color='tab:green')
    ax.set_xscale('log')
    ax.set_title('Residuals around P = a^(3/2)')
    ax.set_xlabel('Semi-major axis [au]')
    ax.set_ylabel('Residual period [yr]')
    ax.grid(alpha=0.3, which='both')
    fig.tight_layout()
    fig.savefig(residual_path, dpi=150)
    plt.close(fig)

    print('线性图像已保存到:', linear_path)
    print('线性 PDF 已保存到:', linear_pdf_path)
    print('对数图像已保存到:', log_path)
    print('残差图像已保存到:', residual_path)


## 图表解读

即使在这组很小的示例数据里，我们也能直接看出：半长轴越大，轨道周期通常越长。在线性坐标下，这种关系更像“远处迅速拉开”；在对数坐标下，它更像一条接近直线的幂律关系。不同坐标选择并不是美术偏好，而是在强调不同的科学结构。

## 图表常见问题提醒

- 没有单位的坐标轴很难复现和比较。
- 只写变量名、不写物理量含义，图会失去教学价值。
- 用默认样式勉强能看，不等于图已经适合科研沟通。
- 若数据跨越多个数量级，对数坐标往往比线性坐标更合适。

## 小结

这一章最重要的不是“会不会调用绘图库”，而是建立图表意识：先做数据检查，再决定变量和坐标尺度，最后用标题、轴标签和单位把图变成真正可解释的论证部件。

In [ ]:
snapshot = {
    'dataset': DATA_PATH.name,
    'n_points': len(rows),
    'mean_kepler_ratio': round(sum((period ** 2) / (axis ** 3) for axis, period in zip(semi_major_axes, orbital_periods)) / len(rows), 3),
    'max_abs_residual_yr': round(max_abs_residual, 3),
    'python_version': platform.python_version(),
}

print('Plotting snapshot:')
for key, value in snapshot.items():
    print(f'  {key}: {value}')


## V2 Evidence Record artifact

本章的记录重点是图表文件、输入数据、坐标轴单位、视觉编码，以及图表能支持和不能支持的 claim。


In [ ]:
artifact_dir = Path('results')
artifact_dir.mkdir(parents=True, exist_ok=True)
evidence_path = artifact_dir / 'ch08_evidence_record.md'

figure_paths = [linear_path, linear_pdf_path, log_path, residual_path]
figure_status = {str(path): path.exists() for path in figure_paths}
mean_kepler_ratio = sum((period ** 2) / (axis ** 3) for axis, period in zip(semi_major_axes, orbital_periods)) / len(rows)
claim_supported = 'larger semi-major axis corresponds to longer orbital period; log-log view makes the power-law structure easier to see'
claim_not_supported = 'this teaching plot cannot test planet formation theory, measurement uncertainty, or non-Solar-System populations'

record_text = f"""# Evidence Record - Ch08 Scientific Figures

## Record Type

- Record type: figure / data check

## 1. Input

- Data / file path, preferably relative to project root: `{DATA_PATH}`
- Data source or generation method: AIforAstronomers teaching planetary orbit table
- Script / notebook: `ch08_plotting_scientific_figures.ipynb`
- Code version / tag, if relevant: fill in the current Git commit or release tag when submitting

## 2. Operation

- Command or entry point: run notebook cells in order
- Key parameters: x-axis semi-major axis in au; y-axis orbital period in years; model `P = a^(3/2)` for residuals
- Random seed, if relevant: not applicable
- Output directory: `../../figures/generated/` for figures; notebook-local `results/` for this Evidence Record

## 3. Output

- Output file(s): `{evidence_path}`; figures `{figure_status}`
- Short result summary: `{len(rows)}` planets; mean `P^2/a^3 = {mean_kepler_ratio:.3f}`; max absolute residual `{max_abs_residual:.3f}` yr
- Generated by which script/notebook: `ch08_plotting_scientific_figures.ipynb`

## 4. Check

- Check performed: data range check, Kepler-ratio check, log-scale comparison, model residual check, figure file existence check
- Evidence from the check: semi-major axis range `{(min(semi_major_axes), max(semi_major_axes))}` au; period range `{(min(orbital_periods), max(orbital_periods))}` yr; figure status `{figure_status}`

## 5. Limit

- Known limitation: this is a tiny Solar System teaching table, not a population-level exoplanet or survey dataset
- Selection / quality / uncertainty issue, if relevant: no measurement uncertainties, selection function, or observational errors are modeled

## 6. Reuse in ML

- Sample / feature / label: one row per planet; orbital quantities could become tabular features in a toy workflow
- Uncertainty / quality flag: no uncertainty or quality flag fields in this teaching table
- Preprocessing record: records axis choices, model residuals, and figure export paths
- Reproducibility evidence: source table, plotting notebook, figure paths, supported claim, and limits are recorded

## Ch08 Fields

- figure file: `{[str(path) for path in figure_paths]}`
- plotting script: `ch08_plotting_scientific_figures.ipynb`
- input data: `{DATA_PATH}`
- axis variables and units: semi-major axis [au], orbital period [yr], residual [yr]
- sample selection: all `{len(rows)}` rows in the teaching table
- visual encoding: scatter points, line model, log-log axes, residual reference line
- claim supported: {claim_supported}
- claim not supported: {claim_not_supported}
"""

evidence_path.write_text(record_text, encoding='utf-8')
print('wrote Evidence Record:', evidence_path)
